# Python Operators — From Basics to Top 1% Understanding

This notebook covers every operator category in Python, then goes past the surface into:

1. Arithmetic, comparison, logical, bitwise, assignment, identity, membership, ternary operators (the basics)
2. **Operator precedence & associativity** — the full table, not just a few examples
3. **Short-circuit evaluation** — why `and`/`or` don't always evaluate both sides, and how to exploit that
4. **Chained comparisons** — Python's `a < b < c` syntax, which is NOT the same as other languages
5. **`==` vs `is`** — nailed down for good
6. **Floor division & modulus with negative numbers** — a classic gotcha
7. **Bitwise operators on negative numbers** — why `~a` isn't what beginners expect
8. **Operator overloading** — how `+`, `==`, `<` etc. actually work via dunder methods
9. Quiz to test yourself

Run each cell — predicting the output before running is the best way to build real intuition.

## 1. Arithmetic Operators

In [1]:
a = 15
b = 4

print("Addition:", a + b)
print("Subtraction:", a - b)
print("Multiplication:", a * b)
print("Division:", a / b)          # always returns a float
print("Floor Division:", a // b)   # rounds toward negative infinity, not zero
print("Modulus:", a % b)
print("Exponentiation:", a ** b)

Addition: 19
Subtraction: 11
Multiplication: 60
Division: 3.75
Floor Division: 3
Modulus: 3
Exponentiation: 50625


### The gotcha almost nobody checks: floor division and modulus with negatives

In C/Java/JavaScript, `//` (integer division) truncates *toward zero*. In Python, `//` rounds *toward negative infinity*. This is a deliberate design choice, and it changes results whenever negative numbers are involved.

In [2]:
print(7 // 2)     #  3  (as expected)
print(-7 // 2)    # -4, NOT -3! Rounds toward -infinity, not toward zero
print(7 // -2)    # -4
print(-7 // -2)   #  3

# The modulus result always has the SAME SIGN as the divisor (unlike C)
print(-7 % 2)     #  1  -- in C this would be -1
print(7 % -2)     # -1

# Python guarantees this identity always holds:
x, y = -7, 2
print("identity check:", x == (x // y) * y + (x % y))

3
-4
-4
3
1
-1
identity check: True


**Why it matters:** if you're porting code from C/Java, or writing anything involving negative-number division (e.g. wrapping array indices, clock arithmetic), assuming truncation-toward-zero will silently produce wrong results in Python.

## 2. Comparison Operators

In [3]:
a = 13
b = 33

print(a > b)
print(a < b)
print(a == b)
print(a != b)
print(a >= b)
print(a <= b)

False
True
False
True
False
True


### Chained comparisons — a Python-specific feature

In most languages, `a < b < c` is parsed as `(a < b) < c` — comparing a boolean to `c`, which is almost always a bug. Python instead treats it as `(a < b) and (b < c)`, evaluating `b` only once.

In [4]:
x = 5
print(1 < x < 10)          # True -- equivalent to (1 < x) and (x < 10)
print(10 < x < 1)          # False

# Compare to what you'd get in a C-like language mentally:
print((1 < x) < 10)        # True < 10  ->  1 < 10  -> True (coincidentally same here, but NOT the same logic)

# Real-world use: range checks in one readable line
age = 25
print("Valid voting age:", 18 <= age <= 120)

True
False
True
Valid voting age: True


## 3. Logical Operators

In [5]:
a = True
b = False
print(a and b)
print(a or b)
print(not a)

False
True
False


### Short-circuit evaluation — and what `and`/`or` actually *return*

This is the detail that separates casual Python use from fluent Python use: `and` and `or` don't just return `True`/`False` — they return one of the **actual operands**, and they stop evaluating as soon as the result is determined.

- `x and y`: if `x` is falsy, return `x` immediately (never evaluates `y`). Otherwise return `y`.
- `x or y`: if `x` is truthy, return `x` immediately (never evaluates `y`). Otherwise return `y`.

In [6]:
print(0 and "never reached")     # 0 is falsy -> returns 0, doesn't touch the string
print("hi" and "world")           # "hi" is truthy -> evaluates and returns "world"
print("" or "default")            # "" is falsy -> returns "default"
print("set" or "never reached")   # "set" is truthy -> short-circuits, returns "set"

# Practical pattern: providing a fallback/default value
user_input = ""
name = user_input or "Guest"
print("name:", name)

# Proof that short-circuiting actually skips the call (not just skips using the result)
def noisy():
    print("  -> noisy() was called!")
    return True

print("Testing False and noisy():")
result = False and noisy()   # noisy() never runs -- no print from inside it
print("result:", result)

0
world
default
set
name: Guest
Testing False and noisy():
result: False


**Practical use of short-circuiting: guard against errors**

In [7]:
data = None

# Safe: if data is None, the left side is falsy, so len(data) is NEVER evaluated -- no crash
if data and len(data) > 0:
    print("has data")
else:
    print("no data (safely avoided calling len(None))")

no data (safely avoided calling len(None))


## 4. Bitwise Operators

In [8]:
a = 10   # 0b1010
b = 4    # 0b0100

print("AND:", a & b)
print("OR: ", a | b)
print("NOT:", ~a)
print("XOR:", a ^ b)
print("Right shift:", a >> 2)
print("Left shift:", a << 2)

AND: 0
OR:  14
NOT: -11
XOR: 14
Right shift: 2
Left shift: 40


### Why `~10` is `-11`, not some positive number

Beginners expect `~a` to just flip bits within, say, 8 bits. But Python integers have **no fixed width** — they're conceptually infinite-precision, using two's-complement semantics. The identity to remember is:

$$\sim x = -x - 1$$

In [9]:
for x in [0, 1, 5, 10, -1, -5]:
    print(f"~{x:>3} = {~x:>4}   (check: -{x}-1 = {-x-1})")

# Visualizing bits of a positive number
print()
print("bin(10):", bin(10))
print("bin(4): ", bin(4))
print("bin(10 & 4):", bin(10 & 4))

~  0 =   -1   (check: -0-1 = -1)
~  1 =   -2   (check: -1-1 = -2)
~  5 =   -6   (check: -5-1 = -6)
~ 10 =  -11   (check: -10-1 = -11)
~ -1 =    0   (check: --1-1 = 0)
~ -5 =    4   (check: --5-1 = 4)

bin(10): 0b1010
bin(4):  0b100
bin(10 & 4): 0b0


## 5. Assignment Operators

In [10]:
a = 10
b = a
print(b)
b += a   # b = b + a
print(b)
b -= a
print(b)
b *= a
print(b)
b <<= a  # left-shift-assign: b = b << a
print(b)

10
20
10
100
102400


### Subtlety: augmented assignment on mutable vs immutable objects

For immutable types (`int`, `str`, `tuple`), `x += y` is just sugar for `x = x + y` — it always rebinds `x` to a new object. But for **mutable** types like `list`, `+=` calls `__iadd__` and mutates **in place** if the type supports it. This is a real, common source of bugs.

In [11]:
# Immutable: += rebinds, no shared-object surprise
s1 = "ab"
s2 = s1
s1 += "c"
print("s1:", s1, " s2:", s2)   # s2 unaffected

# Mutable: += mutates in place -- shared references BOTH see the change
list1 = [1, 2]
list2 = list1
list1 += [3]        # equivalent to list1.extend([3]) -- mutates in place
print("list1:", list1, " list2:", list2)  # list2 changed too!

# Compare to list1 = list1 + [3], which WOULD create a new list
list3 = [1, 2]
list4 = list3
list3 = list3 + [3]  # rebinds list3 to a brand new list
print("list3:", list3, " list4:", list4)  # list4 untouched

s1: abc  s2: ab
list1: [1, 2, 3]  list2: [1, 2, 3]
list3: [1, 2, 3]  list4: [1, 2]


## 6. Identity Operators: `is` / `is not`

In [12]:
a = 10
b = 20
c = a

print(a is not b)
print(a is c)

True
True


### `==` vs `is`, finally nailed down

- `==` calls `__eq__` and asks: **do these have the same value?**
- `is` compares `id()` and asks: **are these literally the same object in memory?**

Two equal values are not necessarily identical objects. The article's line — *"Two variables that are equal do not imply that they are identical"* — is the whole lesson.

In [13]:
x = [1, 2, 3]
y = [1, 2, 3]
print("x == y:", x == y)   # True -- same contents
print("x is y:", x is y)   # False -- two different list objects

# The one case where `is` IS the right tool: comparing against None
value = None
print(value is None)       # Correct, idiomatic, and fast (no __eq__ call needed)
print(value == None)       # Works but considered bad style -- avoid

x == y: True
x is y: False
True
True


## 7. Membership Operators: `in` / `not in`

In [14]:
x = 24
y = 20
my_list = [10, 20, 30, 40, 50]

if x not in my_list:
    print("x is NOT present in given list")
else:
    print("x is present in given list")

if y in my_list:
    print("y is present in given list")
else:
    print("y is NOT present in given list")

x is NOT present in given list
y is present in given list


### Performance: `in` on a `list` vs a `set`/`dict`

`in` works on any iterable, but the *speed* differs enormously by container type:

- `list`/`tuple`: O(n) — scans every element until a match or the end
- `set`/`dict`: O(1) average — uses hashing, just like a dictionary lookup

If you're doing repeated membership checks against a large collection, converting to a `set` first is a real, measurable optimization.

In [15]:
import time

big_list = list(range(1_000_000))
big_set = set(big_list)
target = 999_999   # worst case: near the end

start = time.perf_counter()
_ = target in big_list
list_time = time.perf_counter() - start

start = time.perf_counter()
_ = target in big_set
set_time = time.perf_counter() - start

print(f"list lookup: {list_time*1000:.4f} ms")
print(f"set lookup:  {set_time*1000:.4f} ms")
print(f"set was ~{list_time/set_time:.0f}x faster")

list lookup: 6.6567 ms
set lookup:  0.0304 ms
set was ~219x faster


## 8. Ternary Operator (Conditional Expression)

In [2]:
a, b = 10, 20
minimum = a if a < b else b
print(minimum)

# Can be chained, though readability suffers fast -- use sparingly
score = 75
grade = "A" if score >= 90 else "B" if score >= 75 else "C" if score >= 60 else "F"
print(grade)


if score >= 90:
    grade = "A"
elif score >= 75:
    grade = "B"
elif score >= 60:
    grade = "C"
else:
    grade = "F"

10
B


## 9. Operator Precedence and Associativity

In [17]:
expr = 10 + 20 * 30
print(expr)   # multiplication binds tighter than addition -> 610

name = "Alex"
age = 0

# 'and' binds tighter than 'or', so this reads as: name=="Alex" or (name=="John" and age>=2)
if name == "Alex" or name == "John" and age >= 2:
    print("Hello! Welcome.")
else:
    print("Good Bye!!")

610
Hello! Welcome.


### The full precedence table (highest to lowest)

| Precedence | Operators | Description |
|---|---|---|
| Highest | `()` | Parentheses (grouping) |
| | `**` | Exponentiation (right-associative!) |
| | `+x`, `-x`, `~x` | Unary plus, minus, bitwise NOT |
| | `*`, `/`, `//`, `%` | Multiplication, division, floor division, modulus |
| | `+`, `-` | Addition, subtraction |
| | `<<`, `>>` | Bitwise shifts |
| | `&` | Bitwise AND |
| | `^` | Bitwise XOR |
| | `\|` | Bitwise OR |
| | `==`, `!=`, `<`, `<=`, `>`, `>=`, `is`, `is not`, `in`, `not in` | Comparisons, identity, membership (all equal precedence, chain left-to-right) |
| | `not` | Logical NOT |
| | `and` | Logical AND |
| | `or` | Logical OR |
| Lowest | `x if c else y` | Conditional expression |

**When in doubt, use parentheses.** No one loses points for being explicit, and it prevents exactly the kind of bug in the `name == "Alex" or ...` example above.

### Associativity

In [18]:
print(100 / 10 * 10)   # left-to-right: (100/10)*10 = 100.0
print(5 - 2 + 3)        # left-to-right: (5-2)+3 = 6
print(5 - (2 + 3))      # explicit parens override: 0
print(2 ** 3 ** 2)      # ** is RIGHT-associative: 2 ** (3 ** 2) = 2 ** 9 = 512, NOT (2**3)**2 = 64

100.0
6
0
512


**`**` is the one operator that's right-associative** in the arithmetic group. Every other left/right-grouped arithmetic and comparison operator listed above is left-associative. This single fact explains why `2 ** 3 ** 2` is 512, not 64 — a very common quiz trap.

## 10. Operator Overloading — What's *Really* Happening

Every operator in Python is really a shorthand for calling a **dunder (double-underscore) method** on the object. This is why the *same* `+` symbol can add numbers, concatenate strings, and merge lists — and why you can make your own classes support `+`, `==`, `<`, etc.

In [19]:
print((3).__add__(4))          # 3 + 4, spelled out explicitly
print("ab".__add__("cd"))       # "ab" + "cd"
print([1,2].__add__([3,4]))     # [1,2] + [3,4]
print((5).__lt__(10))           # 5 < 10
print((5).__eq__(5))            # 5 == 5

7
abcd
[1, 2, 3, 4]
True
True


### Defining your own operator behavior

In [20]:
class Money:
    def __init__(self, amount):
        self.amount = amount

    def __add__(self, other):       # defines what '+' does for Money objects
        return Money(self.amount + other.amount)

    def __eq__(self, other):        # defines what '==' does
        return self.amount == other.amount

    def __lt__(self, other):        # defines what '<' does
        return self.amount < other.amount

    def __repr__(self):
        return f"Money(${self.amount})"

wallet1 = Money(50)
wallet2 = Money(30)

print(wallet1 + wallet2)     # calls Money.__add__
print(wallet1 == Money(50))  # calls Money.__eq__
print(wallet2 < wallet1)     # calls Money.__lt__

Money($80)
True
True


**Common dunder methods behind operators:**

| Operator | Dunder method |
|---|---|
| `+` | `__add__` |
| `-` | `__sub__` |
| `*` | `__mul__` |
| `/` | `__truediv__` |
| `//` | `__floordiv__` |
| `%` | `__mod__` |
| `**` | `__pow__` |
| `==` | `__eq__` |
| `<` | `__lt__` |
| `in` | `__contains__` |
| `[]` | `__getitem__` |
| `len()` | `__len__` |

This is why `math.isclose`, `Decimal`, and `Fraction` from the earlier notebook can each define their own comparison behavior, and why a `list` and a `str` can both use `+` and `in` despite being completely different types under the hood: each type implements the dunder methods that make sense for it.

## 11. Quick Self-Check

Predict the output before running.

In [21]:
# Q1
print(-7 // 2)

-4


In [22]:
# Q2
print(2 ** 3 ** 2)

512


In [23]:
# Q3
def side_effect():
    print("called!")
    return True

x = True or side_effect()
print("x:", x)

x: True


In [24]:
# Q4
a = [1, 2]
b = a
a += [3]
print(b)

[1, 2, 3]


---
### Answers

**Q1:** `-4` — Python's `//` floors toward negative infinity, not toward zero.

**Q2:** `512` — `**` is right-associative: `2 ** (3 ** 2)` = `2 ** 9`.

**Q3:** `x: True`, and `"called!"` is **never printed** — `or` short-circuits on the first truthy operand and never evaluates `side_effect()`.

**Q4:** `[1, 2, 3]` — `+=` on a list mutates in place (calls `__iadd__`), so `b`, which points to the same list object, sees the change too.

---
## Summary

| Concept | Beginner takeaway | Top 1% takeaway |
|---|---|---|
| `//` | "Integer division" | Floors toward -infinity, not toward zero — differs from C/Java on negatives |
| `and` / `or` | "Return True/False" | Return one of the actual operands, and short-circuit — skip evaluating the other side entirely |
| `a < b < c` | "Chained comparison" | Evaluates `b` once, equivalent to `(a<b) and (b<c)` — not left-associative re-comparison |
| `~x` | "Flips bits" | `~x == -x - 1`, since Python ints have no fixed width |
| `+=` on lists | "Shorthand for `x = x + y`" | Mutates in place via `__iadd__` — affects every name pointing to that list |
| `==` vs `is` | "Interchangeable" | `==` is value equality (`__eq__`); `is` is identity — use `is` only for `None` |
| Every operator | "Built-in magic" | Shorthand for a dunder method (`__add__`, `__eq__`, ...) — and you can define these yourself |